# 🚀 00 — Environment Setup
## Ekegusii-LLM-Translation · Kineses Cloud / Base Jupyter

> **Run this notebook ONCE after cloning in `start.ipynb`.**

This notebook:
1. Sets working directory to project root
2. Fixes NumPy/Pandas ABI compatibility (critical for Kineses conda)
3. Verifies and auto-installs any missing dependencies
4. Confirms GPU (NVIDIA A100-SXM4-80GB)
5. Verifies Master Corpus & 0% data leakage

---
| Node Spec | Value |
|-----------|-------|
| Python | 3.11.6 (conda-forge) |
| CPU Cores | 22 |
| RAM | 117.9 GB |
| Disk | 967.64 GB |
| GPU | NVIDIA A100-SXM4-80GB (85.1 GB VRAM) |

In [ ]:
# ============================================================
# PATH & REPO AUTO-SYNC BOOSTER — Guarantees latest project code
# ============================================================
import os, sys, site, urllib.request, zipfile

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

home     = os.path.expanduser('~')
proj_dir = os.path.join(home, 'Ekegusii-LLM-Translation-main')
qwen_cfg = os.path.join(proj_dir, 'configs', 'models', 'qwen_7b.yaml')

# Auto-sync if folder is missing OR outdated (lacks qwen_7b.yaml)
if not os.path.isfile(qwen_cfg):
    print('🔄 Outdated or missing repository detected. Auto-syncing latest code from GitHub...')
    zip_path = os.path.join(home, 'repo.zip')
    urllib.request.urlretrieve('https://github.com/aykahsay/Ekegusii-LLM-Translation/archive/refs/heads/main.zip', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(home)
    os.remove(zip_path)
    print('✅ Repository auto-synced to latest main commit!')

if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


In [ ]:
# ============================================================
# CELL 2 — Fix NumPy / Pandas ABI Incompatibility
# ============================================================
import subprocess, sys, importlib

def get_version(pkg):
    try:
        m = importlib.import_module(pkg)
        return getattr(m, '__version__', 'unknown')
    except Exception:
        return 'IMPORT_ERROR'

np_ver  = get_version('numpy')
pd_ver  = get_version('pandas')
print(f'numpy  version: {np_ver}')
print(f'pandas version: {pd_ver}')

needs_fix = (
    pd_ver != 'IMPORT_ERROR'
    and pd_ver.startswith('3')
    and np_ver != 'IMPORT_ERROR'
    and np_ver.startswith('1.')
)
if pd_ver == 'IMPORT_ERROR':
    needs_fix = True

if not needs_fix:
    print('No ABI conflict detected. Skipping fix.')
else:
    print('ABI mismatch detected! Attempting fix...')
    print('  [A] Upgrading numpy >= 2.0...', end=' ')
    rA = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--user', 'numpy>=2.0.0'],
        capture_output=True, text=True
    )
    if rA.returncode == 0:
        print('OK')
        print('numpy upgraded successfully!')
        print('RESTART KERNEL NOW: Kernel -> Restart Kernel, then re-run from Cell 1')
    else:
        print('FAILED (conda read-only)')
        print('  [B] Downgrading pandas to 2.2.3...', end=' ')
        rB = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--user', 'pandas==2.2.3'],
            capture_output=True, text=True
        )
        if rB.returncode == 0:
            print('OK')
            print('pandas downgraded successfully!')
            print('RESTART KERNEL NOW: Kernel -> Restart Kernel, then re-run from Cell 1')
        else:
            print(f'FAILED: {rB.stderr.strip()[:120]}')

In [ ]:
# ============================================================
# CELL 3 — Verify all packages & auto-install missing ones
# ============================================================
print('=' * 60)
print('Verifying dependencies & auto-installing missing packages...')
print('=' * 60)

import importlib, subprocess, sys, site

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

# NOTE: omegaconf/hydra-core are intentionally NOT in this list. This
# project's src/ code no longer depends on them -- src/utils/config_dict.py
# replaced that usage with a minimal PyYAML-based wrapper, because
# omegaconf's antlr4-python3-runtime dependency has no wheel on this host
# and pip cannot complete a source build for it under any flag combination
# tried (--no-build-isolation, --user, explicit setuptools/antlr4 pins).
REQUIRED = [
    'torch', 'pandas', 'numpy', 'matplotlib',
    'transformers', 'datasets', 'evaluate',
    'peft', 'trl', 'bitsandbytes',
    'sacrebleu', 'accelerate', 'tokenizers',
    'rich', 'tqdm', 'scipy', 'sklearn', 'yaml',
]

all_ok = True
for mod in REQUIRED:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'installed')
        print(f'  OK  {mod:<20} {ver}')
    except ImportError:
        pkg_name = 'pyyaml' if mod == 'yaml' else mod
        print(f'  Installing missing package: {pkg_name}...', end=' ', flush=True)
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--user', '--prefer-binary', pkg_name],
            check=False,
        )
        try:
            m = importlib.import_module(mod)
            ver = getattr(m, '__version__', 'installed')
            print(f'OK  ({ver})')
        except ImportError as e:
            print(f'FAILED: {e}')
            all_ok = False

import numpy as np
import pandas as pd
print(f'\n  numpy  : {np.__version__}')
print(f'  pandas : {pd.__version__}')
major_np = int(np.__version__.split(".")[0])
major_pd = int(pd.__version__.split(".")[0])
if (major_pd >= 3 and major_np >= 2) or (major_pd < 3 and major_np < 2):
    print('  ABI match: compatible!')
else:
    print('  WARNING: ABI mismatch! Re-run Cell 2 and restart kernel.')

print('\n✅ All packages verified!' if all_ok else '\n⚠️ Some packages missing!')

In [ ]:
# ============================================================
# CELL 4 — Verify GPU hardware
# ============================================================
import torch

print('=' * 60)
print('GPU Hardware Check')
print('=' * 60)
print(f'  PyTorch Version  : {torch.__version__}')
print(f'  CUDA Available   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'  GPU Name         : {torch.cuda.get_device_name(0)}')
    print(f'  GPU VRAM         : {props.total_memory / 1e9:.1f} GB')
    print(f'  Compute Cap.     : {props.major}.{props.minor}')
    print(f'  CUDA Device      : cuda:{torch.cuda.current_device()}')
    print('\n  A100 Ready for QLoRA fine-tuning!')
else:
    print('\n  No GPU — training will run on CPU only.')

In [ ]:
# ============================================================
# CELL 5 — Verify Master Corpus & 0% Data Leakage
# ============================================================
import os, sys, site
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)
kineses_proj = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(kineses_proj):
    os.chdir(kineses_proj)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.master_corpus.manager import MasterCorpusManager
from src.master_corpus.integrity import DataLeakageChecker

print('=' * 60)
print('Master Corpus Verification')
print('=' * 60)

manager = MasterCorpusManager()
corpus  = manager.load_sentence_corpus()
lexical = manager.load_lexical_corpus()
train   = manager.load_train_split()
val     = manager.load_val_split()
test    = manager.load_test_split()

print(f'  Master Sentence Corpus : {len(corpus):,} multilingual concepts')
print(f'  Master Lexical Corpus  : {len(lexical):,} dictionary entries')
print(f'  Train Split            : {len(train):,} concepts (80%)')
print(f'  Val   Split            : {len(val):,} concepts (10%)')
print(f'  Test  Split            : {len(test):,} concepts (10%)')

print('\nRunning 0% leakage audit...')
checker = DataLeakageChecker(manager)
checker.verify_all()

print('\n' + '=' * 60)
print('  0% DATA LEAKAGE CONFIRMED')
print('  MASTER CORPUS LOADED')
print('  SETUP COMPLETE!')
print('=' * 60)
print('\nOpen any research notebook:')
print('  notebooks/05_instruction_generation.ipynb')
print('  notebooks/07_train_qwen.ipynb')
print('  notebooks/08_train_llama.ipynb')
print('  notebooks/09_translation_evaluation.ipynb')